<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c04-header.png" alt="Nextia Learning · SQL and Data Preparation for AI" width="100%">

# Solution: Validation rules

**[C04-M06-L01 · Validation rules](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/)** · C04, SQL and Data Preparation for AI · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** write down what a dataset promises, as rules that a script checks on every run: required columns, allowed values and ranges, unique keys and expected distributions. First in SQL, then in `validate.py`.

| | |
|---|---|
| **Time** | About 55 minutes |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6. |
| **You should know** | `GROUP BY` and `HAVING`, from [Aggregate correctly](https://learning.nextia-ai.com/courses/sql/m02/aggregate-correctly/). The modelling table and the split, from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/) and [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). |
| **You do not need** | Your local project. The setup below builds the database and the files of Modules 3 to 5 here. |
| **Tested** | 2026-10-07, Python 3.14.6 with pandas 3.0.6, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, script, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/) has the full explanations and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C04/M06-L01-validation-rules/validation-rules-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs `pandas` 3.0.6 in the versions that the course uses. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6
import pandas; print("pandas", pandas.__version__)

## Setup: get the practice data

Run the next cell. It downloads the three files of the Larkfield help-desk export (about 200 KB), checks each file's checksum, and builds the practice database, `larkfield.db`. It works in a folder `ticket-data`, laid out like your project from Module 1, so the code of the lesson runs here unchanged. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: customers 240, tickets 1417, outcomes 1874`. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `ticket-data` (in Colab: the **Files** panel on the left), then run the cell again.

This is the same database that `get_data.py` builds in [Set up a practice database](https://learning.nextia-ai.com/courses/sql/m01/set-up-a-practice-database/#build-the-database). The data is made up for this course; it has no real people.

In [ ]:
# Setup: download the Larkfield export, check it, and build larkfield.db in
# ticket-data/. Standard library only: the same steps as get_data.py.
import csv, hashlib, json, os, sqlite3, subprocess, sys, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C04/"
FILES = {
    "customers.csv": "7b437642f0095524f8311f67a7caeb17100d23eed92618b9cf012cd3d3b79993",
    "tickets.csv": "9c47e08cf2ec5977733d32cc21eb97d0dd01b021d1f5fbf7ec31fef6279b1a28",
    "outcomes.csv": "67b4d9d9275b32061a8888050c351df7314054c11ee1ce29e856c1d5403d0653",
}
# The notebook works in its own folder ticket-data/, marked with a file, so it
# never writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-data").resolve()
PROJECT.mkdir(exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, expected):
    path = PROJECT / "data" / "raw" / name
    if not (path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == expected):
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-data and run again.")
    return path

DB = PROJECT / "larkfield.db"
if not DB.exists():
    paths = {name: fetch(name, sha) for name, sha in FILES.items()}
    build = sqlite3.connect(DB)
    build.executescript("""
        CREATE TABLE customers (customer_id TEXT, segment TEXT, region TEXT, joined_on TEXT);
        CREATE TABLE tickets (ticket_id TEXT, customer_id TEXT, created_at TEXT, channel TEXT, team TEXT,
            priority INTEGER, order_value REAL, word_count INTEGER, first_reply_minutes INTEGER,
            priority_now INTEGER, closed_at TEXT);
        CREATE TABLE outcomes (outcome_id TEXT, ticket_id TEXT, recorded_at TEXT, status TEXT,
            resolution_code TEXT, csat INTEGER);
        CREATE TABLE snapshot (name TEXT, value TEXT);
        INSERT INTO snapshot VALUES ('source', 'Larkfield help-desk export (synthetic, Nextia Learning C04)'),
            ('exported_at', '2026-07-01 06:00:00'), ('data_version', '1.0');
    """)
    for name, path in paths.items():
        with open(path, newline="", encoding="utf-8") as f:
            rows = list(csv.reader(f))
        marks = ", ".join("?" for _ in rows[0])
        build.executemany(f"INSERT INTO {name[:-4]} ({', '.join(rows[0])}) VALUES ({marks})",
                          [[None if v == "" else v for v in row] for row in rows[1:]])
    build.commit()
    build.close()

os.chdir(PROJECT)  # work in the project folder, as in the lesson
for folder in ("queries", "scripts", "reports"):
    Path(folder).mkdir(exist_ok=True)
# Read-only, as q.py: a query cannot change the data by mistake.
con = sqlite3.connect("file:larkfield.db?mode=ro", uri=True)
counts = [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in ("customers", "tickets", "outcomes")]
print("Ready: customers {}, tickets {}, outcomes {}".format(*counts))

## Setup: rebuild the earlier lessons' files

This lesson starts from files that you made in earlier lessons: the extract `data/extract/` from [Reproducible extraction](https://learning.nextia-ai.com/courses/sql/m03/reproducible-extraction/), the clean tables `data/clean/` from [Handle messy values](https://learning.nextia-ai.com/courses/sql/m04/handle-messy-values/), the modelling table `data/model/tickets_model.csv` from [Features and target timing](https://learning.nextia-ai.com/courses/sql/m05/features-and-target-timing/), the train, validation and test sets `data/model/` from [Choose a split](https://learning.nextia-ai.com/courses/sql/m05/choose-a-split/). The next cell downloads the course's finished scripts ([C04/pipeline](https://github.com/usmanahmed99/nextia-learning-labs/tree/main/C04/pipeline)) into `ticket-data` and runs them in order, so you have the same files here. It takes about 20 seconds.

You should see one line per step, ending with `split.py: done`. On your computer, your own scripts from those lessons make the same files.

In [ ]:
# Setup: rebuild the files of the earlier lessons with the course's scripts.
STEPS = ["extract", "clean", "build_features", "split"]
for file in ["queries/extract_tickets.sql", "queries/extract_outcomes.sql"] + [f"scripts/{s}.py" for s in STEPS]:
    if not Path(file).exists():
        download(LABS + "pipeline/" + file, Path(file))
for step in STEPS:
    result = subprocess.run([sys.executable, f"scripts/{step}.py"], capture_output=True, text=True)
    if result.returncode:
        raise SystemExit(f"{step}.py failed:\n{result.stdout}{result.stderr}")
    print(f"{step}.py: done")

## Setup: the helpers

The next cell defines small functions that the lesson uses. You do not need to read them, but you can.

- `sql(query)` runs a query and shows the result as a table, with the number of rows, like `q.py` and the boxes on the lesson page. It shows the first 20 rows; `sql(query, limit=None)` shows all of them.
- `check(query, ...)` compares the result of your query with the expected result, and says what is different. It does not show the answer.
- `expect(what, yours, expected)` compares one of your numbers with the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: show a query result, and check an answer without showing it.
try:
    import pandas as pd
except ImportError:
    pd = None

def run(query, db=None):
    cursor = (db or con).execute(query)
    columns = [d[0] for d in cursor.description] if cursor.description else []
    return columns, cursor.fetchall()

def sql(query, db=None, limit=20):
    """Run a query and show its result as a table, with the row count."""
    columns, rows = run(query, db)
    if not columns:
        print("No result: the cell has no SELECT yet. Write your query between the triple quotes, then run the cell again.")
        return
    shown = rows if limit is None else rows[:limit]
    if pd is not None:
        # Show NULL as NULL, as q.py and the boxes on the lesson page do.
        table = pd.DataFrame(shown, columns=columns, dtype=object)
        display(table.where(table.notna(), "NULL"))
    else:
        print("  ".join(columns))
        for row in shown:
            print("  ".join("NULL" if v is None else str(v) for v in row))
    more = f" (showing {len(shown)}; use limit=None to see all)" if len(shown) < len(rows) else ""
    print(f"{len(rows)} row{'s' if len(rows) != 1 else ''}{more}")

def fingerprint(rows, ordered=False):
    cells = [tuple("NULL" if v is None else f"{v:.6f}" if isinstance(v, float) else str(v) for v in r) for r in rows]
    return hashlib.sha256(repr(cells if ordered else sorted(cells)).encode()).hexdigest()[:16]

def check(query, rows, columns, expected, ordered=False, db=None):
    """Compare your result with the expected one: the column count, the row
    count, then the values (as a fingerprint, so the answer stays hidden)."""
    try:
        got_columns, got = run(query, db)
    except sqlite3.Error as error:
        print(f"Not yet: the query has an error: {error}")
        return
    if len(got_columns) != columns:
        print(f"Not yet: your result has {len(got_columns)} columns; the task asks for {columns}.")
    elif len(got) != rows:
        print(f"Not yet: your result has {len(got)} rows; the task expects {rows}.")
    elif fingerprint(got, ordered) != expected:
        print("Not yet: the number of rows is right, but some values" + (" or their order" if ordered else "") + " are different.")
    else:
        print("Check passed.")

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    same = abs(yours - expected) <= tolerance * max(1, abs(expected)) if isinstance(expected, float) else yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours}, which is not the expected value.")

## Setup: run a script

In the lesson, you run each script in a terminal, for example `python scripts/validate.py`. Here, the function `python()` does the same. It runs the script with the Python of this notebook, prints the output, and then prints the **exit code**: 0 means success, any other number means failure.

Run the cell. It prints nothing. It also imports `json`, to read the JSON reports of the pipeline.

In [ ]:
import json

def python(*args, cwd=None):
    """Run a Python script as `python <args>` in a terminal. Print its output and its exit code."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True, cwd=cwd)
    print((result.stdout + result.stderr).rstrip())
    print(f"[exit code {result.returncode}]")
    return result

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A promise about the data

Priya will train a model on Mei's files. She asks: "What can I rely on?" Mei answers with rules.

- The **schema** of a table is the list of its columns and the type of each column.
- A **data contract** is an agreement between the maker of a dataset and its users. It has the schema and the rules for the values.
- A **validation rule** is one check from the contract.

Four kinds of rule: **required values**, **allowed values and ranges**, a **unique key**, and an **expected distribution** (for example, 5% to 30% of the tickets are escalated).

The next cell makes five tiny tickets. Find the problems before you run it: one `Email`, one priority 5, one ticket twice, one missing channel.

In [ ]:
tiny = pd.DataFrame({
    "ticket_id": ["T-1", "T-2", "T-3", "T-3", "T-4"],
    "channel": ["email", "Email", "chat", "chat", None],
    "priority": [2, 1, 5, 5, 3],
    "escalated_72h": [0, 1, 0, 0, 0],
})
tiny

Each rule below finds the rows that **break** it. A rule passes when it finds no row.

> **Predict.** Which rows does each rule find? Is the escalation rate inside 5% to 30%? Then run the cell.

In [ ]:
print("required values:", list(tiny.loc[tiny["channel"].isna(), "ticket_id"]))
print("allowed values: ", list(tiny.loc[tiny["channel"].notna() & ~tiny["channel"].isin(["email", "chat", "phone", "web_form"]), "ticket_id"]))
print("range 1 to 3:   ", list(tiny.loc[~tiny["priority"].between(1, 3), "ticket_id"]))
print("unique key:     ", list(tiny.loc[tiny["ticket_id"].duplicated(keep=False), "ticket_id"]))
print("escalation rate:", tiny["escalated_72h"].mean())

> **Check.** You should see `['T-4']`, `['T-2']`, `['T-3', 'T-3']`, `['T-3', 'T-3']` and `0.2`. The rate, 20%, is inside 5% to 30%: that rule passes.

The first three rules look at single rows. The rate rule looks at the whole table: it catches a bug where every row looks fine but the total is wrong. None of these problems raises an error. A rule makes a quiet mistake loud.

## 2. Rules as queries that must return no rows

A useful way to write a rule: a query that returns the rows that break it. The rule passes with **0 rows**. When it fails, you see the bad rows, not only "false".

These queries run on the **raw** export, the `tickets` table in `larkfield.db`. The raw export is not clean, so most rules find problems. After the pipeline, the same rules must find nothing.

**Required columns.** `pragma_table_info('tickets')` lists the table's columns. The query lists each expected column that is not there. Run the cell.

In [ ]:
sql("""
WITH expected(name) AS (
    VALUES ('ticket_id'), ('customer_id'), ('created_at'), ('channel'),
           ('team'), ('priority'), ('order_value'), ('word_count')
)
SELECT name AS missing_column
FROM expected
WHERE name NOT IN (SELECT name FROM pragma_table_info('tickets'));
""")

> **Check.** You should see an empty table and `0 rows`: every expected column exists.

> **Your turn: change one thing.** In the cell above, change `'channel'` to `'contact_channel'` and run it again. You should see one row, `contact_channel`. The last lesson of this module has a new export that renames `channel` in this way.

**Unique key.** A **primary key** is a column whose value is different in every row. The export does not guarantee one, so you check it.

> **Predict.** How many ticket IDs appear more than once? You found them in Module 1. Then run the cell.

In [ ]:
sql("""
SELECT ticket_id, COUNT(*) AS copies
FROM tickets
GROUP BY ticket_id
HAVING COUNT(*) > 1
ORDER BY ticket_id;
""")

> **Check.** You should see 14 rows, from T-10031 to T-11291, each with 2 copies: the exact duplicate rows from the export page overlap. `clean.py` removes them.

## 3. Your turn: catch every channel that is not allowed

The starting query looks correct, but it has a quiet mistake.

> **Predict.** How many groups does it return? Is the missing channel among them?

Run the cell. You see 4 groups: ` chat`, `EMAIL`, `Email` and `webform`. The 38 rows with no channel are not there. For a NULL channel, `channel NOT IN (...)` is not true, it is unknown, so `WHERE` drops the row.

> **Your turn.** Change the query so that it also returns the NULL group. Then run the check cell. When it is correct, you see 5 groups and `Check passed.`

In [ ]:
allowed_channels = """
SELECT channel, COUNT(*) AS n
FROM tickets
WHERE channel IS NULL
   OR channel NOT IN ('email', 'chat', 'phone', 'web_form')
GROUP BY channel
ORDER BY n DESC;
"""
sql(allowed_channels)

In [ ]:
check(allowed_channels, rows=5, columns=2, expected="dbaaeb82bbae20f2")

Five groups, 78 rows together. The first group is `NULL`. A rule about allowed values must say what it does with a missing value. Here, missing is not allowed.

## 4. Ranges

A range rule gives the smallest and the largest allowed value. Grace says that an order above $2,000 is very rare, and a negative order is not possible.

> **Predict.** Which two problems from Module 4 does this rule find? Then run the cell.

In [ ]:
sql("""
SELECT ticket_id, created_at, channel, order_value
FROM tickets
WHERE order_value < 0
   OR order_value > 2000
ORDER BY order_value;
""", limit=None)

> **Check.** You should see 36 rows. The first 11 have `-1`, the "unknown" sentinel. The other 25 are January web-form tickets, in cents. The last of them, T-10147, is also the ticket with the year 2036.

> **Your turn.** The contract says: `word_count` is 0 to 10,000. Zero is allowed, because a message can be an attachment only. Write a query that returns `ticket_id` and `word_count` for every row that breaks this rule. Then run the check cell.

In [ ]:
word_count_rule = """
SELECT ticket_id, word_count
FROM tickets
WHERE word_count < 0
   OR word_count > 10000;
"""
sql(word_count_rule)

In [ ]:
check(word_count_rule, rows=1, columns=2, expected="a9166ba019020c98")

One row: T-10361 with 48,210 words, the pasted log file. `clean.py` puts it in quarantine with the reason `word_count_above_10000`.

## 5. All the rules in one report

`UNION ALL` puts the results of several `SELECT` statements one under the other. This query counts the problems of each rule. On your computer, it is `queries/60-contract-checks.sql`.

> **Predict.** Which rule passes on the raw export? Then run the cell.

In [ ]:
contract_checks = """
-- Each rule counts the rows that break it. A rule passes with 0.
SELECT 'ticket_id is unique' AS rule,
       (SELECT COUNT(*) FROM (SELECT ticket_id FROM tickets
                              GROUP BY ticket_id HAVING COUNT(*) > 1)) AS problems
UNION ALL
SELECT 'required values are present', COUNT(*)
FROM tickets
WHERE ticket_id IS NULL OR created_at IS NULL OR team IS NULL OR priority IS NULL
UNION ALL
SELECT 'channel has an allowed value', COUNT(*)
FROM tickets
WHERE channel IS NULL OR channel NOT IN ('email', 'chat', 'phone', 'web_form')
UNION ALL
SELECT 'order_value is 0 to 2000', COUNT(*)
FROM tickets
WHERE order_value < 0 OR order_value > 2000
UNION ALL
SELECT 'word_count is 0 to 10000', COUNT(*)
FROM tickets
WHERE word_count < 0 OR word_count > 10000
UNION ALL
SELECT 'created_at is in the first half of 2026', COUNT(*)
FROM tickets
WHERE created_at < '2026-01-01' OR created_at >= '2026-07-01';
"""
sql(contract_checks)

> **Check.** You should see 14, 0, 78, 36, 1 and 1. Only "required values are present" passes. The other five find the problems that Modules 3 to 5 handle. That is the point: the rules describe the data **after** the pipeline.

## 6. The contract in validate.py

Priya receives CSV files, not the database. So Mei writes the contract again in Python, for the files in `data/model/`: `scripts/validate.py`. It uses pandas only. The [lesson page](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/#the-contract-in-validatepy) explains each part.

Each of the next four cells writes one part of the script. `%%writefile` writes the cell into a file; `%%writefile -a` adds the cell to the end of the file. The code is the same as in the lesson.

**Part 1: the contract as data.** The rules are plain Python values at the top of the file. A change to the contract is a change to these lines. Run the cell. You should see `Writing scripts/validate.py`.

In [ ]:
%%writefile scripts/validate.py
"""Check the delivered dataset against its contract: the rules that Priya can
rely on in data/model/. Prints PASS or FAIL for each rule, writes
reports/validation.json, and exits with code 1 if any rule fails."""

import json
import sys
from pathlib import Path

import pandas as pd

MODEL = Path("data/model")
REPORT = Path("reports/validation.json")
TIME_FORMAT = "%Y-%m-%d %H:%M:%S"
SPLITS = ["train", "valid", "test"]

# ---------------------------------------------------------------- the contract

# Every column, its type, and whether a value is required.
COLUMNS = {
    "ticket_id": ("text", True),
    "customer_id": ("text", False),            # guests have no customer
    "created_at": ("datetime", True),
    "channel": ("text", True),
    "team": ("text", True),
    "priority": ("integer", True),
    "segment": ("text", True),
    "region": ("text", True),
    "order_value": ("number", False),          # no order (account) or unknown
    "word_count": ("integer", True),
    "customer_tenure_days": ("number", False),  # guests and deleted customers
    "prior_tickets_90d": ("integer", True),
    "created_hour": ("integer", True),
    "escalated_72h": ("integer", True),        # the target
}
# Known only after the ticket is created: never in the dataset.
NEVER = ["priority_now", "first_reply_minutes", "closed_at"]
ALLOWED = {
    "channel": {"email", "chat", "phone", "web_form", "unknown"},
    "team": {"delivery", "returns", "payment", "warranty", "account"},
    "segment": {"home", "trade", "business", "guest", "unknown"},
    "region": {"north", "south", "east", "west", "unknown"},
}
RANGES = {  # inclusive; None means no limit
    "priority": (1, 3),
    "order_value": (0, 2000),
    "word_count": (0, 10_000),
    "customer_tenure_days": (0, None),
    "prior_tickets_90d": (0, None),
    "created_hour": (0, 23),
    # From the start of the selection to the snapshot minus 72 hours.
    "created_at": (pd.Timestamp("2026-01-01"), pd.Timestamp("2026-06-28 06:00:00")),
}
ESCALATION_RATE = (0.05, 0.30)

**Part 2: small helpers.** `rule()` prints `PASS` or `FAIL` with the problems under it. `load()` reads every value as **text**, because pandas chooses types quietly: one `$10.85` turns the whole column into text, with no warning. `convert()` then tries each value. Run the cell. You should see `Appending to scripts/validate.py`.

In [ ]:
%%writefile -a scripts/validate.py

# ---------------------------------------------------------------- helpers

results = []


def rule(name, problems):
    """Record one rule. An empty list of problems means the rule passes."""
    results.append({"rule": name, "passed": not problems, "problems": problems})
    print(f"{'PASS' if not problems else 'FAIL'}  {name}")
    for problem in problems:
        print(f"      {problem}")


def first(df, mask, column):
    """The first bad row, as an example: its ticket ID and its value."""
    row = df[mask].iloc[0]
    value = row[column]
    shown = repr(value) if isinstance(value, str) else value
    return f"first: {row['ticket_id']}, {shown}"


def n(count, word):
    """'1 row', '3 rows'."""
    return f"{count} {word}" + ("" if count == 1 else "s")


def load(path):
    """Read every value as text. Types are checked by the rules, not assumed."""
    return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[""])


def convert(text, kind):
    """Convert a text column to its type. A value that cannot be read becomes missing."""
    if kind == "text":
        return text
    if kind == "datetime":
        return pd.to_datetime(text, format=TIME_FORMAT, errors="coerce")
    numbers = pd.to_numeric(text, errors="coerce")
    if kind == "integer":
        numbers = numbers.where(numbers % 1 == 0)  # 2.5 is not an integer
    return numbers

**Part 3: rules for the modelling table.** Each block is one rule, and each rule finds the bad rows first. `duplicated(keep=False)` marks **every** copy of a repeated ID. The allowed-values rule skips missing values: they are the job of the required-values rule. Run the cell.

In [ ]:
%%writefile -a scripts/validate.py

# ---------------------------------------------------------------- the rules

def check_table(raw):
    """Rules for the modelling table, one row per ticket. Returns the typed table."""
    missing = [c for c in COLUMNS if c not in raw.columns]
    leaky = [c for c in raw.columns if c in NEVER]
    extra = [c for c in raw.columns if c not in COLUMNS and c not in NEVER]
    rule("required columns are present",
         [f"missing column: {c}" for c in missing])
    rule("no column that is known only after creation",
         [f"{c} must not be in the dataset" for c in leaky] + [f"unexpected column: {c}" for c in extra])
    if missing:
        return None

    df = raw.copy()
    problems = []
    for column, (kind, _) in COLUMNS.items():
        df[column] = convert(raw[column], kind)
        bad = raw[column].notna() & df[column].isna()
        if bad.any():
            problems.append(f"{column}: not a valid {kind} in {n(bad.sum(), 'row')} ({first(raw, bad, column)})")
    rule("every value has its column's type", problems)

    problems = []
    for column, (_, required) in COLUMNS.items():
        if required and raw[column].isna().any():
            problems.append(f"{column}: missing in {n(raw[column].isna().sum(), 'row')}")
    rule("no missing values in required columns", problems)

    repeated = df["ticket_id"].duplicated(keep=False)
    rule("ticket_id is unique",
         [f"{n(df.loc[repeated, 'ticket_id'].nunique(), 'ticket ID')} repeated "
          f"(first: {df.loc[repeated, 'ticket_id'].iloc[0]})"] if repeated.any() else [])

    problems = []
    for column, allowed in ALLOWED.items():
        bad = df[column].notna() & ~df[column].isin(allowed)
        if bad.any():
            values = sorted(df.loc[bad, column].unique())
            problems.append(f"{column}: values not allowed in {n(bad.sum(), 'row')}: {values}")
    rule("categories use allowed values only", problems)

    problems = []
    for column, (low, high) in RANGES.items():
        values = df[column]
        bad = pd.Series(False, index=df.index)
        if low is not None:
            bad |= values < low
        if high is not None:
            bad |= values > high
        if bad.any():
            problems.append(f"{column}: outside {low} to {high} in {n(bad.sum(), 'row')} ({first(df, bad, column)})")
    rule("numbers and dates are in their allowed ranges", problems)

    bad = ~df["escalated_72h"].isin([0, 1])
    rule("the target escalated_72h is 0 or 1",
         [f"not 0 or 1 in {n(bad.sum(), 'row')} ({first(df, bad, 'escalated_72h')})"] if bad.any() else [])
    return df

**Part 4: rules for the splits, and the result.** A ticket in two splits is a leak. Splits out of time order mix the future into training. The rate rule runs on every split, because one broken split can hide in a good total. `sys.exit(1)` ends the script with exit code 1. Run the cell.

In [ ]:
%%writefile -a scripts/validate.py


def rate_problem(name, target):
    """The escalation rate must be in the expected range, or something upstream changed."""
    low, high = ESCALATION_RATE
    rate = pd.to_numeric(target).mean()
    if not low <= rate <= high:
        return [f"{name}: escalation rate {rate:.1%} is outside {low:.0%} to {high:.0%}"]
    return []


def check_splits(table, parts):
    """Rules for train, valid and test."""
    empty = [name for name, part in parts.items() if len(part) == 0]
    rule("every split has rows", [f"{name} has no rows" for name in empty])
    if empty:
        return

    problems = []
    for name, part in parts.items():
        if list(part.columns) != list(table.columns):
            problems.append(f"{name}: the columns are not the same as in tickets_model.csv")
    together = pd.concat(parts.values())
    if len(together) != len(table):
        problems.append(f"the splits have {len(together)} rows together; tickets_model.csv has {len(table)}")
    if set(together["ticket_id"]) != set(table["ticket_id"]):
        problems.append("the splits do not contain the same tickets as tickets_model.csv")
    rule("together, the splits are exactly the modelling table", problems)

    problems = []
    for i, a in enumerate(SPLITS):
        for b in SPLITS[i + 1:]:
            shared = set(parts[a]["ticket_id"]) & set(parts[b]["ticket_id"])
            if shared:
                problems.append(f"{a} and {b} share {n(len(shared), 'ticket ID')} (first: {sorted(shared)[0]})")
    rule("no ticket is in two splits", problems)

    problems = []
    for a, b in zip(SPLITS, SPLITS[1:]):
        last = pd.to_datetime(parts[a]["created_at"], format=TIME_FORMAT).max()
        start = pd.to_datetime(parts[b]["created_at"], format=TIME_FORMAT).min()
        if not last < start:
            problems.append(f"{a} ends at {last}, which is not before the start of {b} at {start}")
    rule("the splits are in time order: train, then valid, then test", problems)

    problems = rate_problem("all rows", table["escalated_72h"])
    for name, part in parts.items():
        problems += rate_problem(name, part["escalated_72h"])
    rule(f"escalation rate between {ESCALATION_RATE[0]:.0%} and {ESCALATION_RATE[1]:.0%}", problems)


def main():
    files = ["tickets_model.csv"] + [f"{name}.csv" for name in SPLITS]
    missing = [f for f in files if not (MODEL / f).exists()]
    if missing:
        print(f"FAIL  missing file(s) in {MODEL}: {', '.join(missing)}")
        print("Run build_features.py and split.py first.")
        sys.exit(1)

    print(f"Checking {MODEL} against the contract")
    table = check_table(load(MODEL / "tickets_model.csv"))
    if table is not None:
        check_splits(table, {name: load(MODEL / f"{name}.csv") for name in SPLITS})

    failed = [r for r in results if not r["passed"]]
    REPORT.parent.mkdir(exist_ok=True)
    REPORT.write_text(json.dumps({"files": files, "passed": not failed, "rules": results}, indent=2) + "\n",
                      encoding="utf-8")
    if failed:
        print(f"{len(failed)} of {len(results)} rules failed. Fix the step that made the data, then run it again.")
        sys.exit(1)
    print(f"All {len(results)} rules passed.")


if __name__ == "__main__":
    main()

Now run the script on the files from Module 5.

> **Predict.** How many rules does the script check? Do they all pass? Then run the cell.

In [ ]:
contract = python("scripts/validate.py")

> **Check.** You should see 13 `PASS` lines, `All 13 rules passed.` and `[exit code 0]`. The script also saved the results in `reports/validation.json`, the **quality report** that Priya receives with the data. Run the next cell to see its first rule.

In [ ]:
report = json.loads(Path("reports/validation.json").read_text(encoding="utf-8"))
print("passed:", report["passed"], "| rules:", len(report["rules"]))
report["rules"][0]

## 7. What the distribution rule catches

A rule that never fails proves nothing. Mei tests the distribution rule with a real bug: in `build_features.py`, she types `minutes` instead of `hours`.

The next cell makes this bug in `build_features.py`, then runs `build_features.py` and `split.py`. Both run with no error.

> **Predict.** Which rules of `validate.py` fail? Then run the cell, and the one after it.

In [ ]:
features = Path("scripts/build_features.py")
correct = features.read_text(encoding="utf-8")
features.write_text(correct.replace("pd.Timedelta(hours=72)", "pd.Timedelta(minutes=72)"), encoding="utf-8")
print([line for line in features.read_text(encoding="utf-8").splitlines() if line.startswith("WINDOW")])
for step in ["scripts/build_features.py", "scripts/split.py"]:
    print(step, "exit code", subprocess.run([sys.executable, step], capture_output=True).returncode)

In [ ]:
bug = python("scripts/validate.py")

> **Check.** You should see two `FAIL` lines. `created_at` is outside its range in 29 rows (first: T-11374). The escalation rate is 0.8% for all rows, and 0.5%, 1.8% and 0.8% in the splits. The last lines are `2 of 13 rules failed.` and `[exit code 1]`.

Every row is still valid: each target is 0 or 1. But few tickets reach a senior agent within 72 **minutes**. The bug also moved the cut-off for a final target, so the 29 tickets that are too recent came back. Without the rules, Priya trains a model on the wrong target, and its scores look normal.

Mei changes the line back. Run the cell: it restores `build_features.py` and runs the two steps again.

In [ ]:
features.write_text(correct, encoding="utf-8")
for step in ["scripts/build_features.py", "scripts/split.py"]:
    print(step, "exit code", subprocess.run([sys.executable, step], capture_output=True).returncode)
print([line for line in features.read_text(encoding="utf-8").splitlines() if line.startswith("WINDOW")])

> **Check.** You should see exit code 0 twice and `WINDOW = pd.Timedelta(hours=72)`.

## 8. Guided practice: break the contract on purpose

This cell adds the last train row to the end of `test.csv`, as step 5 of the lesson's guided practice does.

> **Predict.** Which rules fail? Then run the cell, and the one after it.

In [ ]:
train = pd.read_csv("data/model/train.csv")
train.tail(1).to_csv("data/model/test.csv", mode="a", header=False, index=False)
print("added", train.tail(1)["ticket_id"].iloc[0], "to test.csv")

In [ ]:
broken = python("scripts/validate.py")

> **Check.** You should see 3 `FAIL` lines. The splits have 1370 rows together; `tickets_model.csv` has 1369. Train and test share 1 ticket ID (T-10929). Valid ends at 2026-05-31 20:57:12, which is not before the start of test at 2026-04-30 23:54:47. The rate rule still passes. The last lines are `3 of 13 rules failed.` and `[exit code 1]`.

One wrong row broke three rules, and each message points to it from a different side. One row does not change a rate much. That is why a contract needs row rules **and** distribution rules.

Repair the files: run the step that made them, then the contract again. Run the cell. You should see `All 13 rules passed.` and `[exit code 0]`.

In [ ]:
repair = subprocess.run([sys.executable, "scripts/split.py"], capture_output=True)
print("split.py exit code", repair.returncode)
contract = python("scripts/validate.py")

## 9. Your turn: a rule for the size of the evaluation sets

Priya asks for a new rule: "I want at least 30 escalated tickets in valid and in test. With fewer, a score changes a lot from one ticket to the next."

> **Your turn.** Complete the function `min_escalated_problems(parts)`. `parts` has the three splits, as `validate.py` loads them (every value is text). For `"valid"` and for `"test"`, count the escalated tickets. If a count is below `MIN_ESCALATED`, add a message such as `"test: 35 escalated tickets; at least 40 are needed"`. Run the cell, then the check cell.

> **Predict.** With 30, does the rule pass? With 40?

In [ ]:
MIN_ESCALATED = 30


def min_escalated_problems(parts):
    """The rule: valid and test have at least MIN_ESCALATED escalated tickets each."""
    problems = []
    for name in ["valid", "test"]:
        escalated = int(pd.to_numeric(parts[name]["escalated_72h"]).sum())
        if escalated < MIN_ESCALATED:
            problems.append(f"{name}: {escalated} escalated tickets; at least {MIN_ESCALATED} are needed")
    return problems

In [ ]:
parts = {name: pd.read_csv(f"data/model/{name}.csv", dtype=str, keep_default_na=False, na_values=[""])
         for name in ["train", "valid", "test"]}
chosen = MIN_ESCALATED
results_by_minimum = {}
for MIN_ESCALATED in (30, 40):
    results_by_minimum[MIN_ESCALATED] = min_escalated_problems(parts)
MIN_ESCALATED = chosen
with_30, with_40 = results_by_minimum[30], results_by_minimum[40]
if with_30 == [] and len(with_40) == 1 and with_40[0].startswith("test") and "35" in with_40[0]:
    print("Check passed.")
elif with_30:
    print(f"Not yet: with MIN_ESCALATED = 30, your function finds {len(with_30)} problem(s). Count the escalated tickets again.")
else:
    print(f"Not yet: with MIN_ESCALATED = 40, your function finds {len(with_40)} problem(s); the expected number is different, "
          "or the message does not name the split and the count.")

Now put the rule into `validate.py`, at the end of `check_splits()`, as in the lesson's answer. The next cell does it for you: `add_rule(30)` writes `MIN_ESCALATED = 30` in the contract part, adds your function, and adds the line `rule(f"valid and test have at least {MIN_ESCALATED} escalated tickets each", ...)`. Then it runs the script.

Run the cell. You should see a 14th rule and `All 14 rules passed.` If your function is not complete yet, the new rule finds no problem, so it passes here and in the next cell too.

In [ ]:
import inspect

CONTRACT_13 = globals().get("CONTRACT_13") or Path("scripts/validate.py").read_text(encoding="utf-8")

def add_rule(minimum):
    """Write validate.py with the 13 rules plus your rule, with MIN_ESCALATED = minimum."""
    last_rule = 'rule(f"escalation rate between {ESCALATION_RATE[0]:.0%} and {ESCALATION_RATE[1]:.0%}", problems)\n'
    new_rule = '    rule(f"valid and test have at least {MIN_ESCALATED} escalated tickets each", min_escalated_problems(parts))\n'
    script = CONTRACT_13.replace("ESCALATION_RATE = (0.05, 0.30)\n", f"ESCALATION_RATE = (0.05, 0.30)\nMIN_ESCALATED = {minimum}\n")
    script = script.replace(last_rule, last_rule + new_rule)
    script = script.replace("\n\ndef main():", "\n\n" + inspect.getsource(min_escalated_problems) + "\n\ndef main():")
    Path("scripts/validate.py").write_text(script, encoding="utf-8")

add_rule(30)
contract = python("scripts/validate.py")

> **Predict.** Now with 40: which split fails, and what does the message say? Then run the cell.

In [ ]:
add_rule(40)
contract = python("scripts/validate.py")

> **Check.** You should see `FAIL  valid and test have at least 40 escalated tickets each`, the problem `test: 35 escalated tickets; at least 40 are needed`, and `1 of 14 rules failed.` Valid has 45 escalated tickets, so it passes.

The data is not wrong here: the test set is small. The fix is a decision for Priya and Mei, for example a longer test period. A rule can fail because the data does not meet a need. That is useful too: it shows the problem before anyone trusts a score.

Run the cell to go back to 30. You should see `All 14 rules passed.`

In [ ]:
add_rule(30)
contract = python("scripts/validate.py")

## 10. Failure case: a rule that cannot fail

**Symptom:** the rule for allowed channels passes on the extract, where you know that `Email` and `webform` exist.

**Cause:** the allowed values come from the data that the rule checks. Run the cell.

In [ ]:
df = pd.read_csv("data/extract/tickets.csv", dtype=str, keep_default_na=False, na_values=[""])
allowed = set(df["channel"].dropna())  # the mistake: the rule comes from the data
print(sorted(allowed))
print("rows that break the rule:", (~df["channel"].dropna().isin(allowed)).sum())

> **Check.** You should see eight spellings, from `' chat'` to `'webform'`, and `rows that break the rule: 0`. Every value is "allowed", because the list is made from the data. The rule can never fail.

**Fix:** write the allowed values by hand, from the agreement with the people who know the data. Grace knows the four channels. Then test each new rule once on data that must break it. Run the cell.

In [ ]:
contract_channels = {"email", "chat", "phone", "web_form"}  # the fix: the rule comes from the agreement
print("rows that break the rule:", (~df["channel"].dropna().isin(contract_channels)).sum())

> **Check.** You should see `rows that break the rule: 40`: the 20 `Email`, 8 ` chat`, 7 `EMAIL` and 5 `webform` rows of the extract.

**In AI work:** a check that always passes is worse than no check, because people trust it. A new category in production, for example a channel `social`, then reaches the model with nobody noticing.

## Check your understanding and recap

Answer the **knowledge checks** on the [lesson page](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/). They count toward your certificate when you are signed in and enrolled.

In this lesson you wrote a data contract as rules: required values, allowed values and ranges, unique keys and expected distributions. Each rule returns the rows that break it. `validate.py` checks 13 rules, writes `reports/validation.json` and exits with code 1 on failure. You tested the rules on data that must break them.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Track lineage](https://learning.nextia-ai.com/courses/sql/m06/track-lineage/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/sql/m06/validation-rules/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.